# este notebook preprocesa las imagenes para ser usado en uno posterior

In [ ]:
# EXP (no es del baseline): Kaggle monta los inputs en otra carpeta cuando el notebook se crea por API.
# Arma /tmp/lab/input/<nombre> -> carpeta real y se para en /tmp/lab/work, asi '../input/<nombre>/...' del baseline funciona sin tocarlo.
import os, glob
def _buscar(nombre, archivo):
    for prof in range(0, 5):
        patron = "/kaggle/input/" + "*/" * prof
        for d in glob.glob(patron):
            d = d.rstrip("/")
            if os.path.basename(d).endswith(nombre) and os.path.exists(os.path.join(d, archivo)):
                return d
    raise FileNotFoundError(f"{nombre}/{archivo} no aparece en /kaggle/input: {sorted(glob.glob('/kaggle/input/*/*'))[:50]}")
_inputs = {"petfinder-adoption-prediction": "train/train.csv"}
if not all(os.path.exists(f"../input/{n}/{a}") for n, a in _inputs.items()):
    os.makedirs("/tmp/lab/input", exist_ok=True)
    os.makedirs("/tmp/lab/work", exist_ok=True)
    for n, a in _inputs.items():
        destino = f"/tmp/lab/input/{n}"
        if not os.path.exists(destino):
            os.symlink(_buscar(n, a), destino)
    os.chdir("/tmp/lab/work")
print({n: os.path.realpath(f"../input/{n}") for n in _inputs})

In [ ]:
# Importación de Librerías
import os 

import numpy as np
import pandas as pd

from tqdm import tqdm

from tensorflow.keras.utils import load_img, img_to_array
from tensorflow.keras.applications.efficientnet import EfficientNetB0, preprocess_input

In [ ]:
model = EfficientNetB0(weights='imagenet', include_top=False, pooling="max")

pics = [f for f in os.listdir('../input/petfinder-adoption-prediction/train_images') if f.endswith("-1.jpg")]
train_img  = pd.DataFrame([], columns=[f"img_feat_{i + 1}" for i in range(1280)])
for p in tqdm(pics):
    img = load_img(f'../input/petfinder-adoption-prediction/train_images/{p}')
    img = img_to_array(img)
    img = np.expand_dims(img, axis=0)
    img = preprocess_input(img)
    petId = p[:-len("-1.jpg")]  # strip("-1.jpg") borraba tambien los 1 al principio y al final del PetID
    train_img.loc[petId] = model.predict(img)[0]

In [ ]:
train_img

In [ ]:
train_img.to_parquet("train.parquet")

In [ ]:
pics = [f for f in os.listdir('../input/petfinder-adoption-prediction/test_images') if f.endswith("-1.jpg")]
test_img  = pd.DataFrame([], columns=[f"img_feat_{i + 1}" for i in range(1280)])
for p in tqdm(pics):
    img = load_img(f'../input/petfinder-adoption-prediction/test_images/{p}')
    img = img_to_array(img)
    img = np.expand_dims(img, axis=0)
    img = preprocess_input(img)
    petId = p[:-len("-1.jpg")]  # strip("-1.jpg") borraba tambien los 1 al principio y al final del PetID
    test_img.loc[petId] = model.predict(img)[0]

In [ ]:
test_img

In [ ]:
test_img.to_parquet("test.parquet")

In [ ]:
# si la celda de inputs cambio de carpeta, dejar los parquet en el output de Kaggle
import shutil
if os.getcwd() != "/kaggle/working":
    for f in ("train.parquet", "test.parquet"):
        shutil.copy(f, "/kaggle/working/" + f)
